# SmolLM 3B — Generalized Medical QA Fine-Tune (ChatDoctor 10k)

Fine-tunes **`HuggingFaceTB/SmolLM3-3B`** with **QLoRA** on a curated, stratified
10,000-record subset of the ChatDoctor dataset (17 medical domains: cardiology, neurology,
respiratory, GI, orthopedics, dermatology, ENT, psychiatry, gynecology/obstetrics, urology,
pediatrics, endocrinology, infectious disease, ophthalmology, dental, oncology, sexual
health). This is a **general-purpose medical QA assistant** — plain instruction→response
tuning, no RAG context slot, no domain-specific system prompt.

**Pipeline:**
1. Environment setup
2. Hugging Face authentication
3. Load & clean the ChatDoctor 10k dataset
4. Format into an instruction-tuning prompt + train/val split
5. Save the formatted dataset to disk (for the multi-GPU training script)
6. Write & launch the multi-GPU (DDP) training script
7. Merge the LoRA adapter into the base model
8. Push adapter + merged model to Hugging Face Hub
9. Quick sanity-check generation

> Environment: Kaggle Notebook, **dual T4 GPUs, true data-parallel (DDP) training** — each
> GPU holds its own full 4-bit copy of the model and processes a different batch shard,
> syncing gradients every step. (Note: `device_map="auto"` model-sharding across GPUs does
> **not** give a speedup here — it only ever keeps one GPU busy at a time. DDP is what
> actually uses both GPUs in parallel.)


## 1. Environment check

In [1]:
!nvidia-smi
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  cuda:{i} ->", torch.cuda.get_device_name(i))


Sat Jul 25 16:15:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   54C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q -U "transformers>=4.51.0" "accelerate>=1.4.0" "peft>=0.14.0" \
    "bitsandbytes>=0.45.0" "trl>=0.15.0" "datasets>=3.2.0" "huggingface_hub>=0.27.0" \
    sentencepiece


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 98.9 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 50.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 885.0/885.0 kB 43.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 771.9/771.9 kB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 66.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.2/119.2 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 98.5 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 32.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not inst

## 2. Hugging Face authentication

Add your token as a **Kaggle Secret**:
1. Go to https://huggingface.co/settings/tokens → **New token** → role **Write** → copy it.
2. In the Kaggle notebook: **Add-ons → Secrets → Add secret**, name it `HF_TOKEN`, paste the value.

`SmolLM3-3B` is Apache-2.0 licensed and ungated, so — unlike a gated model — there's no
separate license-acceptance step needed, just the write-access token above.


In [3]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
login(token=HF_TOKEN)
print("Logged in to Hugging Face Hub.")


Logged in to Hugging Face Hub.


## 3. Load & clean the ChatDoctor 10k dataset

Upload `chatdoctor_curated_10k.json` as a Kaggle Dataset (or attach it under this
notebook's Input) and update the path below to match.


In [4]:
from datasets import load_dataset
import re

# <-- adjust this path to match wherever you attached the dataset on Kaggle
DATA_PATH = "/kaggle/input/datasets/naziaa02/chatdoctor10k/chatdoctor_curated_10k (1).json"

raw = load_dataset("json", data_files=DATA_PATH, split="train")
print(raw)
print(raw[0])


Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['instruction', 'input', 'output'],
    num_rows: 10000
})
{'instruction': "If you are a doctor, please answer the medical questions based on the patient's description.", 'input': 'hi. Im abheesh. 19 years old boy. Due to sexual excitement i had made sex with my girl friend. Inorder to avoid pregnancy, i released my penis just before ejaculation.. But my friends are saying that, it.s too risky. So i gave her contraceptive pill within 24 hours. Now im little tensed about unwanted pregnancy. Is there any possibility remaining for that.? Help me.. Is that releasing technique effective to prevent pregnancy?', 'output': "Hi, I think there is very little chance of pregnancy as you have taken two measures for that. Even then if you have doubt. If she doesn't get her periods on time, get a urine pregnancy test done at home.  If negative, no need to worry.  If positive, consult a doctor if you don't want pregnancy.  Hope I have answered your question.  If you have any ot

In [5]:
# --- Cleaning: whitespace normalization + dedup + length filtering ---
# (dataset was already curated/stratified upstream, so this mainly guards against
# whitespace noise and any accidental exact-duplicate rows.)
def clean_row(ex):
    return {
        "instruction": ex["instruction"].strip(),
        "input": re.sub(r"\s+", " ", ex["input"]).strip(),
        "output": re.sub(r"\s+", " ", ex["output"]).strip(),
    }

cleaned = raw.map(clean_row)

seen = set()
def is_valid_unique(ex):
    key = ex["input"].lower()
    if key in seen:
        return False
    seen.add(key)
    n_in, n_out = len(ex["input"].split()), len(ex["output"].split())
    return 3 <= n_in <= 700 and 5 <= n_out <= 700

cleaned = cleaned.filter(is_valid_unique)
print(f"Rows before cleaning: {len(raw)} -> after cleaning/dedup: {len(cleaned)}")


Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

Filter:   0%|          | 0/10000 [00:00<?, ? examples/s]

Rows before cleaning: 10000 -> after cleaning/dedup: 10000


## 4. Format into an instruction-tuning prompt + train/val split

In [6]:
PROMPT_TEMPLATE = """### Instruction:
{instruction}

### Patient Query:
{input}

### Response:
{output}"""

def format_example(ex):
    return {
        "text": PROMPT_TEMPLATE.format(
            instruction=ex["instruction"], input=ex["input"], output=ex["output"]
        )
    }

formatted = cleaned.map(format_example, remove_columns=cleaned.column_names)
formatted = formatted.shuffle(seed=42)
print(formatted[0]["text"][:600])


Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

### Instruction:
If you are a doctor, please answer the medical questions based on the patient's description.

### Patient Query:
I m 87, have long-since been unable to attain an erection, think I recall I had prostate removed, twice removed cancerous portions of colon, ending up with a Colostomy ( which I handle well and with which I have no problems .. in short, this old guy ( me ) thinks ( though he will check with his GP and other MDs ) it s time to quit my one-a-day Tamsulosin / Flomax. I read where dizziness and related side effects can be connected.

### Response:
Thanks for your questi


In [7]:
split_ds = formatted.train_test_split(test_size=0.08, seed=42)
train_dataset, eval_dataset = split_ds["train"], split_ds["test"]
print("Train:", len(train_dataset), " Eval:", len(eval_dataset))


Train: 9200  Eval: 800


## 5. Save the formatted dataset to disk

The training script in the next section runs as a **separate multi-process job**
(`accelerate launch --multi_gpu`), so it can't read the `train_dataset` / `eval_dataset`
Python objects living in this kernel's memory — we save them to disk once here, and the
script loads them back inside each of the two DDP processes.


In [8]:
MODEL_ID = "HuggingFaceTB/SmolLM3-3B"

DATA_DIR = "/kaggle/working/sm_dataset"
train_dataset.save_to_disk(f"{DATA_DIR}/train")
eval_dataset.save_to_disk(f"{DATA_DIR}/eval")
print("Saved formatted dataset to", DATA_DIR)


Saving the dataset (0/1 shards):   0%|          | 0/9200 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/800 [00:00<?, ? examples/s]

Saved formatted dataset to /kaggle/working/sm_dataset


## 6. Write & launch the multi-GPU (DDP) training script

We write a standalone `train.py` and launch it with `accelerate launch --multi_gpu
--num_processes 2`, so **both T4s run true data-parallel (DDP) training** — each GPU holds
its own 4-bit-quantized copy of SmolLM 3B and processes a different batch shard, syncing
gradients every step.

Key correctness details baked in (same pattern as the MedGemma notebook):
- `device_map={"": device_string}` so each DDP process places its bnb 4-bit model on
  **its own** GPU (mixing bitsandbytes 4-bit + naive `device_map="auto"` + DDP does not work).
- `ddp_find_unused_parameters=False` for memory/speed.
- `gradient_accumulation_steps` halved from 4 → 2 versus the single-GPU version, so the
  **effective global batch size stays the same** (4 × 2 accum × 2 GPUs = 16, same as
  4 × 4 accum × 1 GPU = 16 before) — you get the 2-GPU speedup without changing training
  dynamics or needing to re-tune the learning rate.
- fp16 (not bf16) throughout — T4 is Turing-generation and has no bf16 tensor cores.


In [9]:
%%writefile /kaggle/working/train.py
import torch
from datasets import load_from_disk
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    EarlyStoppingCallback,
)
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTConfig, SFTTrainer
from accelerate import PartialState

MODEL_ID = "HuggingFaceTB/SmolLM3-3B"
DATA_DIR = "/kaggle/working/sm_dataset"
OUTPUT_DIR = "/kaggle/working/outputs"

# Each DDP process gets its own GPU index via accelerate's PartialState
device_string = PartialState().process_index

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# T4 is Turing-generation and has no bf16 tensor cores -- fp16 (not bf16) everywhere.
COMPUTE_DTYPE = torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map={"": device_string},   # <-- one full 4-bit copy per GPU, NOT split across GPUs
    dtype=COMPUTE_DTYPE,
    attn_implementation="sdpa",
)
model.config.use_cache = False

if device_string == 0:
    footprint_gb = model.get_memory_footprint() / 1e9
    print(f"[rank0] SmolLM 3B memory footprint: {footprint_gb:.2f} GB (expect ~2-3GB in 4-bit)")

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

train_dataset = load_from_disk(f"{DATA_DIR}/train")
eval_dataset = load_from_disk(f"{DATA_DIR}/eval")

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,        # effective batch = 4 * 2 * 2 GPUs = 16 (same as before)
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    num_train_epochs=3,
    max_length=768,
    dataset_text_field="text",
    packing=False,                        # no flash-attn on T4, so packing stays off (correctness > speed)
    dataloader_num_workers=2,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=50,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=True,
    bf16=False,
    ddp_find_unused_parameters=False,
    report_to="none",
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    peft_config=lora_config,
    args=training_args,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

# fp16/AMP safety net: the GradScaler needs fp32 master weights for whatever is trainable
# (the LoRA adapters); frozen 4-bit base weights are untouched since requires_grad is False.
cast_count = 0
for name, p in trainer.model.named_parameters():
    if p.requires_grad and p.dtype != torch.float32:
        p.data = p.data.to(torch.float32)
        cast_count += 1
if cast_count and device_string == 0:
    print(f"[rank0] Cast {cast_count} trainable parameter(s) to fp32 for AMP compatibility")

trainer.train()

if trainer.is_world_process_zero():
    trainer.model.save_pretrained(f"{OUTPUT_DIR}/lora_adapter")
    tokenizer.save_pretrained(f"{OUTPUT_DIR}/lora_adapter")
    print("Saved LoRA adapter to", f"{OUTPUT_DIR}/lora_adapter")


Writing /kaggle/working/train.py


In [10]:
# Launch true 2-GPU DDP training as a separate process group (required for
# `accelerate launch --multi_gpu` to work correctly -- it can't run inline in this kernel).
# mixed_precision is fp16 here (not bf16) to match train.py -- T4 has no bf16 tensor cores.
!accelerate launch --multi_gpu --num_processes 2 --mixed_precision fp16 /kaggle/working/train.py


The following values were not passed to `accelerate launch` and had defaults used instead:
	`--num_machines` was set to a value of `1`
	`--dynamo_backend` was set to a value of `'no'`
To avoid this warning pass in values for each of the problematic parameters or run `accelerate config`.
config.json: 100%|█████████████████████████| 1.92k/1.92k [00:00<00:00, 1.34MB/s]
tokenizer_config.json: 100%|███████████████| 50.4k/50.4k [00:00<00:00, 37.2MB/s]

tokenizer.json: downloading bytes: ▍                        |  293kB            
tokenizer.json: downloading bytes: █████████████████████████| 4.24MB,  420kB/s  
tokenizer.json: reconstructing file: 100%|█████████| 17.2MB / 17.2MB, 1.71MB/s  
special_tokens_map.json: 100%|█████████████████| 289/289 [00:00<00:00, 1.35MB/s]
chat_template.jinja: 100%|█████████████████| 5.60k/5.60k [00:00<00:00, 14.3MB/s]
model.safetensors.index.json: 100%|████████| 26.9k/26.9k [00:00<00:00, 77.6MB/s]
Reconstructing (incomplete total...): |            |  0.00B /  

## 7. Merge the LoRA adapter into the base model

In [12]:
ADAPTER_DIR = "/kaggle/working/outputs/lora_adapter"
MERGED_DIR = "/kaggle/working/merged_model"

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

# peft's LoRA-injection code checks the installed `torchao` version (used only for a
# different quantization backend we never use in this pipeline) and raises ImportError
# if it's older than 0.16.0. Kaggle's base image ships 0.10.0, and since a plain pip
# upgrade doesn't retroactively fix an already-imported module in a running kernel,
# we monkey-patch the check itself to just say "not available" -- which is the correct
# answer for this pipeline anyway, since we only ever use bitsandbytes 4-bit + LoRA,
# never torchao layers. This makes dispatch_torchao() skip straight to the next
# dispatcher (the one that actually handles our Linear layers).
import peft.tuners.lora.torchao as _peft_lora_torchao
_peft_lora_torchao.is_torchao_available = lambda: False

# tokenizer/model were only ever loaded inside the train.py subprocess above, not in this
# kernel -- load the tokenizer back from the adapter dir (train.py saved it there).
tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float16,
    device_map="auto",
    attn_implementation="sdpa",
)

merged = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
merged = merged.merge_and_unload()

merged.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)
print("Merged model saved to", MERGED_DIR)

Loading weights:   0%|          | 0/326 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged model saved to /kaggle/working/merged_model


## 8. Push to Hugging Face Hub

Pushes both the lightweight LoRA adapter and the full merged model, plus an
auto-generated model card. Reuses the `HF_TOKEN` login from Section 2.


In [13]:
HF_USERNAME = "jubayer009"          # <-- change this
ADAPTER_REPO = f"{HF_USERNAME}/huggingface-smolLm3b-lora"
MERGED_REPO = f"{HF_USERNAME}/huggingface-smolLm3b"

from huggingface_hub import HfApi

api = HfApi()
api.create_repo(ADAPTER_REPO, private=True, exist_ok=True)
api.create_repo(MERGED_REPO, private=True, exist_ok=True)

api.upload_folder(repo_id=ADAPTER_REPO, folder_path=ADAPTER_DIR)
api.upload_folder(repo_id=MERGED_REPO, folder_path=MERGED_DIR)

card_text = f"""---
base_model: {MODEL_ID}
tags:
- smollm3
- medical-qa
- chatdoctor
- lora
- general-purpose
license: apache-2.0
---

# {MERGED_REPO.split('/')[-1]}

Fine-tuned from `{MODEL_ID}` on a curated, stratified 10,000-record subset of the
ChatDoctor dataset spanning 17 medical domains (cardiology, neurology, respiratory, GI,
orthopedics, dermatology, ENT, psychiatry, gynecology/obstetrics, urology, pediatrics,
endocrinology, infectious disease, ophthalmology, dental, oncology, sexual health).

A **general-purpose medical question-answering assistant**: plain instruction-tuned
patient-query -> doctor-style response, no RAG context slot, no domain-specific system prompt.

## Intended use
General preliminary medical guidance / triage-style responses across a broad range of
everyday medical complaints.

## Limitations & safety
Not a diagnostic tool, not a substitute for a licensed physician, and not validated for
clinical use. May produce incorrect, incomplete, or outdated medical information -- always
direct end-users to consult a qualified healthcare professional.
"""
with open("/kaggle/working/README.md", "w") as f:
    f.write(card_text)
api.upload_file(path_or_fileobj="/kaggle/working/README.md", path_in_repo="README.md", repo_id=MERGED_REPO)
api.upload_file(path_or_fileobj="/kaggle/working/README.md", path_in_repo="README.md", repo_id=ADAPTER_REPO)

print("Pushed adapter to:", ADAPTER_REPO)
print("Pushed merged model to:", MERGED_REPO)


Pushed adapter to: jubayer009/huggingface-smolLm3b-lora
Pushed merged model to: jubayer009/huggingface-smolLm3b


## 9. Quick sanity check (optional)

One single generation call to confirm the merged model actually loads and responds sensibly
— not a full chat UI, just a smoke test.


In [15]:
test_query = "I have had a persistent dry cough and mild fever for the past 3 days. What could this be?"

prompt = PROMPT_TEMPLATE.format(
    instruction=raw[0]["instruction"],
    input=test_query,
    output="",
).rsplit("### Response:", 1)[0] + "### Response:\n"

inputs = tokenizer(prompt, return_tensors="pt").to(merged.device)
with torch.no_grad():
    output_ids = merged.generate(
        **inputs,
        max_new_tokens=300,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
        pad_token_id=tokenizer.pad_token_id,
    )

decoded = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print(decoded.split("### Response:")[-1].strip())


Thanks for your question on HealLM. I can understand your concern. Persistent cough with fever can be due to many reasons. 1. Bronchitis. 2. Tuberculosis. 3. Pneumonia. 4. Lungs infection (pneumococcal pneumonia). So better to consult pulmonologist and get done clinical examination of respiratory system and PFT (Pulmonary Function Test). PFT is needed to rule out bronchitis. Tuberculosis and pneumonia need CT thorax and sputum examination for the diagnosis. So consult pulmonologist and discuss all these. Hope I have solved your query. I will be happy to help you further. Wishing good health to your son. Thanks.
